# 04 · Model comparison (both profiles)

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from boxbox.config import load_config
from boxbox.paths import artifact_paths, dataset_paths
from boxbox.preprocessing.build import read_lap_dataset

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)
COLOURS = {"SOFT": "#c93a55", "MEDIUM": "#b07c00", "HARD": "#3b6fb0", "UNKNOWN": "#77766f"}
subset, full = load_config("subset"), load_config("full")

In [ ]:
metrics = json.loads(artifact_paths('subset').combined_metrics.read_text())['profiles']
rows = [dict(profile=p, **m) for p, s in metrics.items() for m in s['metrics'] if m['split'] != 'train']
table = pd.DataFrame(rows).pivot_table(index=['target', 'model'], columns=['profile', 'split'], values='mae')
table.round(3)

Selected model per profile (lowest validation next-lap MAE) and leave-one-race-out CV on the training races:

In [ ]:
pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s['loro_cv_summary']]).pivot(index='model', columns='profile', values='mean').round(3)

In [ ]:
{p: s['selected_model'] for p, s in metrics.items()}

In [ ]:
h = pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s['horizon']])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, profile in zip(axes, ['subset', 'full']):
    for (model, split), g in h[h.profile == profile].groupby(['model', 'split']):
        ax.plot(g.k, g.mae, marker='o', ls='-' if split == 'val' else '--', label=f'{model} {split}')
    ax.set(title=f'{profile}: MAE vs laps ahead', xlabel='k'); ax.legend(fontsize=7)
fig.tight_layout()

Interpretable parameters: polynomial ridge coefficients (seconds) and gradient-boosting permutation importances (validation MAE increase).

In [ ]:
poly = json.loads((artifact_paths('subset').models / 'poly_ridge.meta.json').read_text())['coefficients']
pd.Series(poly).filter(like='tyre_age').round(4)

In [ ]:
hgb = json.loads((artifact_paths('full').models / 'hgb.meta.json').read_text())['importances']
pd.Series(hgb).sort_values(ascending=False).round(4)

In [ ]:
pd.DataFrame([{**r, 'profile': p} for p, s in metrics.items() for r in s['ablation_deg_features'] + s['fuel_sensitivity']])

In [ ]:
from IPython.display import Image
Image(str(artifact_paths('full').figures / 'pred_vs_actual.png'))